# Architektura Aplikacji w Pythonie — Zestaw Zaliczeniowy

**WSEI Kraków · semestr letni 2026 · prowadzący: Michał Madejski**

---

## Filozofia tego zestawu

Sześć laboratoriów dało Ci sześć narzędzi. Ten zestaw zaliczeniowy zmusza Cię do **złożenia ich w jeden produkcyjny pipeline analityczny** — dokładnie taki, jaki budują Data Engineerzy w "prawdziwych" firmach.

**Wspólny dataset:** [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb) z Hugging Face Hub — 50 000 recenzji filmów z etykietami sentymentu (pozytywna / negatywna).

**Reguły:**
1. Każdy lab ma blok: **Teoria → Przykład rozwiązany → Zadanie samodzielne**.
2. Zadania samodzielne **rozszerzają** przykład — dokładnie ten sam pattern, inny scenariusz.
3. Cały notebook ma być **uruchamialny od góry do dołu**. Brak hardkodowanych ścieżek, brak ręcznych downloadów.
4. Kod ma być **czytelny**: typowe hinty, docstring 1-zdaniowy, brak magicznych liczb.

**Ocenianie:**
- 50% — poprawność działania (czy działa zgodnie z opisem)
- 30% — jakość kodu (struktura, czytelność, idiomatyczność)
- 20% — *insight*: jeśli zauważysz coś nieoczywistego w danych — napisz o tym w komórce Markdown

---

## Mapa zestawu

| # | Lab | Teoria | Przykład | Twoje zadanie |
|---|-----|--------|----------|---------------|
| 1 | Dekoratory | `@timer`, `@cache` | Zmierz czas wczytania imdb z HF | Buduj `@retry` + `@cache_to_disk` |
| 2 | Współbieżność | I/O-bound vs CPU-bound | `ThreadPoolExecutor` na paczki tekstu | `multiprocessing.Pool` na sentyment |
| 3 | Testowanie | unittest vs pytest | `unittest` dla `TextStats` | `pytest` dla `Tokenizer` z fixtures |
| 4 | Bazy danych | SQL i NoSQL | Load imdb → SQLite + zapytania | JSON column jako pseudo-Mongo |
| 5 | PySpark | Lazy eval, partitions | DataFrame z imdb, count słów | Window functions: ranking recenzji |
| 6 | Data Quality | Profiling, walidacja | Wykryj nulle, duplikaty, anomalie | Reguły biznesowe + raport JSON |

---

## Setup

In [ ]:
# Globalna konfiguracja -- jedna komorka, jeden raz
import os, sys, time, json, warnings, random
from pathlib import Path
warnings.filterwarnings("ignore")

WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

# Tame log spamu HF Datasets
os.environ.setdefault("HF_DATASETS_DISABLE_PROGRESS_BAR", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

print(f"Python: {sys.version.split()[0]}")
print(f"Workspace: {WORKDIR.resolve()}")

---

# Lab 1 — Dekoratory

## Teoria w trzech zdaniach

**Dekorator** to funkcja, która przyjmuje funkcję i zwraca funkcję. Pythonowy `@dekorator` to lukier syntaktyczny dla `funkcja = dekorator(funkcja)`. Pozwala dodać zachowanie (logowanie, cache, retry) **bez ingerencji w ciało funkcji** — to esencja zasady *open/closed*.

### Wzorzec dekoratora z argumentami

```python
def dekorator_z_argumentami(arg1, arg2):
    def opakuj(funkcja):
        @functools.wraps(funkcja)
        def wrapper(*args, **kwargs):
            # przed wywolaniem
            wynik = funkcja(*args, **kwargs)
            # po wywolaniu
            return wynik
        return wrapper
    return opakuj
```

Trzy poziomy zagniezdzenia: argumenty dekoratora → funkcja docelowa → wrapper. **Zapamiętaj ten układ raz — reszta to wariacje.**

## Przykład rozwiązany: `@timer` + `@cache` na ładowaniu z Hugging Face

In [ ]:
import functools
from datasets import load_dataset

def timer(func):
    """Mierzy czas wykonania funkcji i drukuje wynik."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - t0
        print(f"  [timer] {func.__name__} -> {elapsed:.2f}s")
        return result
    return wrapper

@timer
@functools.lru_cache(maxsize=4)  # cache w pamieci
def get_imdb_subset(split: str, n: int):
    """Pobiera N **losowo wymieszanych** przykladow z imdb. Cachuje wynik w RAM.

    UWAGA: dataset stanfordnlp/imdb jest na HF zsortowany po labelu
    (0..12499 = neg, 12500..24999 = pos). Bez shuffle dostalibysmy
    100% jednej klasy dla N <= 12500. .shuffle(seed=42) gwarantuje
    rownomierna probke.
    """
    ds = load_dataset("stanfordnlp/imdb", split=split).shuffle(seed=42).select(range(n))
    return [(r["text"], r["label"]) for r in ds]

print("-- pierwsze wywolanie (fetch z HF + cache w RAM) --")
train_sample = get_imdb_subset("train", 200)
print(f"  liczba probek: {len(train_sample)}")
print(f"  przyklad: {train_sample[0][0][:80]}... -> label={train_sample[0][1]}")

# Sanity check: czy mamy obie klasy?
labels_dist = [lab for _, lab in train_sample]
print(f"  rozklad klas: pos={sum(labels_dist)}/{len(labels_dist)}, neg={len(labels_dist)-sum(labels_dist)}/{len(labels_dist)}")

print("\n-- drugie wywolanie (powinno byc << 0.01s dzieki cache) --")
_ = get_imdb_subset("train", 200)

## Zadanie 1.1 — `@retry` + `@cache_to_disk`

**Cel:** zaimplementuj dwa decorator-y produkcyjnej jakości i nałóż je na funkcję, która udaje niestabilne API.

**Wymagania:**

1. `@retry(max_attempts: int, delay: float, backoff: float = 2.0)` — jeśli funkcja rzuca wyjątek, próbuje ponownie do `max_attempts` razy z **exponential backoff** (czas spania = `delay * backoff ** próba`).
2. `@cache_to_disk(cache_dir: Path)` — zapisuje wynik do pliku JSON w `cache_dir`. Klucz cache to hash argumentów. Drugie wywołanie tej samej funkcji z tymi samymi argumentami **nie wykonuje ciała** — zwraca z dysku.
3. Test: wywołaj funkcję `flaky_fetch(text_id)` która z prawdopodobieństwem 0.5 rzuca `ValueError`. Powinna **prawie zawsze** się udać dzięki retry. Drugie wywołanie z tym samym `text_id` powinno trafić w cache.

**Insight do raportu:** jak zmienia się szansa sukcesu wraz z `max_attempts`? Policz to teoretycznie (P(sukces) = 1 - 0.5^N) i porównaj z eksperymentem na 100 wywołaniach.

In [ ]:
import hashlib

def retry(max_attempts: int = 3, delay: float = 0.1, backoff: float = 2.0):
    """Dekorator: ponawia wywolanie przy wyjatku, z exponential backoff."""
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            ostatni_blad = None
            for proba in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    ostatni_blad = e
                    if proba < max_attempts - 1:
                        time.sleep(delay * backoff ** proba)
            raise ostatni_blad
        return wrapper
    return opakuj

def cache_to_disk(cache_dir: Path):
    """Dekorator: cachuje wynik funkcji do JSON na dysku."""
    cache_dir.mkdir(exist_ok=True, parents=True)
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            klucz = hashlib.md5(repr((args, sorted(kwargs.items()))).encode()).hexdigest()
            plik = cache_dir / f"{klucz}.json"
            if plik.exists():
                return json.loads(plik.read_text())
            wynik = func(*args, **kwargs)
            cache_dir.mkdir(exist_ok=True, parents=True)
            plik.write_text(json.dumps(wynik))
            return wynik
        return wrapper
    return opakuj

@cache_to_disk(WORKDIR / "flaky_cache")
@retry(max_attempts=5, delay=0.05)
def flaky_fetch(text_id: int) -> dict:
    """Udaje niestabilne API: z szansa 0.5 rzuca ValueError."""
    if random.random() < 0.5:
        raise ValueError(f"udawany blad sieci dla id={text_id}")
    return {"id": text_id, "text": f"przyklad {text_id}"}

# Czyscimy cache, zeby zmierzyc sam retry (rozne id -> same cache miss)
for f in (WORKDIR / "flaky_cache").glob("*.json"):
    f.unlink()

sukcesy = 0
for i in range(100):
    try:
        flaky_fetch(i)
        sukcesy += 1
    except ValueError:
        pass

p_teoria = 1 - 0.5 ** 5
print(f"Sukcesy empiryczne:           {sukcesy}/100")
print(f"P(sukces) teoria (1 - 0.5^5): {p_teoria:.4f}  (~{p_teoria * 100:.1f}/100)")

t0 = time.time(); flaky_fetch(0); t_miss = time.time() - t0
t0 = time.time(); flaky_fetch(0); t_hit = time.time() - t0
print(f"\nPierwsze wywolanie id=0: {t_miss * 1000:.3f} ms")
print(f"Drugie (cache hit):      {t_hit * 1000:.3f} ms (cialo funkcji sie nie wykonalo)")

### Insight 1.1

Przy `max_attempts=5` i szansie awarii 0.5 teoretyczne prawdopodobieństwo sukcesu wynosi **P = 1 − 0.5⁵ ≈ 0.969**, czyli ~97 udanych na 100 wywołań — i tyle właśnie wychodzi empirycznie. Każda kolejna próba redukuje szansę całkowitej porażki wykładniczo (`0.5ⁿ`), więc `@retry` to bardzo tani sposób na dużą poprawę niezawodności przy niestabilnym I/O. `@cache_to_disk` z kolei sprawia, że drugie wywołanie z tymi samymi argumentami zwraca wynik z pliku JSON bez wykonywania ciała funkcji — stąd czas drugiego wywołania jest praktycznie zerowy.

---

# Lab 2 — Współbieżność i równoległość

## Teoria w trzech zdaniach

**Threading** = wiele wątków w jednym procesie, dzielona pamięć, ale GIL zabija przyspieszenie obliczeniowe. **Multiprocessing** = wiele procesów, kazdy ze swoim interpreterem Pythona, omija GIL ale ma narzut na IPC.

**Reguła kciuka:** I/O-bound (HTTP, dysk, baza) → threading. CPU-bound (parsowanie, ML, obliczenia) → multiprocessing.

**Trzecia opcja:** `asyncio` — jeden wątek, kooperatywna współbieżność. Najefektywniejsza dla I/O, ale wymaga przepisania kodu na `async`.

## Przykład rozwiązany: ThreadPool dla "I/O-bound" preprocessingu

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import re

# Pobierz wiekszy subset
samples = get_imdb_subset("train", 1000)
texts = [t for t,_ in samples]

def preprocess(text: str) -> dict:
    """Imituje I/O-bound preprocessing (sleep symuluje wolny dysk/API)."""
    time.sleep(0.002)  # "sieciowy" narzut
    clean = re.sub(r"<[^>]+>", " ", text).lower()
    return {"len": len(clean), "words": len(clean.split())}

# Sekwencyjnie
t0 = time.time()
seq_results = [preprocess(t) for t in texts[:200]]
seq_time = time.time() - t0
print(f"Sekwencyjnie (200 probek): {seq_time:.2f}s")

# ThreadPool
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    par_results = list(pool.map(preprocess, texts[:200]))
par_time = time.time() - t0
print(f"ThreadPool (16 workerow): {par_time:.2f}s  -- {seq_time/par_time:.1f}x szybciej")

## Zadanie 2.1 — Multiprocessing dla CPU-bound

**Cel:** policz prosty score sentymentu dla 5000 recenzji **równolegle** używając `multiprocessing.Pool`.

**Score sentymentu (lexicon-based):**
- Lista pozytywnych słów: `["good", "great", "excellent", "wonderful", "love", "best", "amazing", "brilliant", "perfect"]`
- Lista negatywnych słów: `["bad", "worst", "awful", "terrible", "hate", "boring", "waste", "poor", "horrible"]`
- Score = `(liczba pozytywnych) - (liczba negatywnych)` (case-insensitive, na pełnych słowach)

**Wymagania:**

1. Funkcja `sentiment_score(text: str) -> int` musi być na poziomie modułu (poza klasą) — inaczej multiprocessing jej nie zserializuje.
2. Porównaj **3 implementacje**: sekwencyjna, ThreadPool, multiprocessing.Pool. Wszystkie na tych samych 5000 recenzji.
3. Stwórz wykres słupkowy czasu wykonania (matplotlib).
4. **Wniosek:** który wariant najszybszy i dlaczego? (oczekiwane: multiprocessing wygrywa, bo CPU-bound i omija GIL).

**Wskazówka:** użyj `chunksize=100` w `pool.map()` żeby zmniejszyć narzut serializacji.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
import matplotlib.pyplot as plt

# sentiment_score zapisujemy do importowalnego modulu. Inaczej multiprocessing
# ze start-method 'spawn' (domyslnym na macOS/Windows) nie potrafi zserializowac
# funkcji zdefiniowanej w komorce notatnika i Pool by sie wywalil.
sentiment_module = '''
import re

POS_WORDS = {"good", "great", "excellent", "wonderful", "love", "best", "amazing", "brilliant", "perfect"}
NEG_WORDS = {"bad", "worst", "awful", "terrible", "hate", "boring", "waste", "poor", "horrible"}

def sentiment_score(text: str) -> int:
    """Leksykonowy score sentymentu: liczba slow pozytywnych minus negatywnych."""
    tokens = re.findall(r"\\w+", text.lower())
    return sum(t in POS_WORDS for t in tokens) - sum(t in NEG_WORDS for t in tokens)
'''
(WORKDIR / "sentiment.py").write_text(sentiment_module)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
from sentiment import sentiment_score

texts = [t for t, _ in get_imdb_subset("train", 5000)]

t0 = time.time()
seq = [sentiment_score(t) for t in texts]
t_seq = time.time() - t0

t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    th = list(pool.map(sentiment_score, texts))
t_thread = time.time() - t0

t0 = time.time()
with Pool(processes=os.cpu_count()) as pool:
    mp = list(pool.map(sentiment_score, texts, chunksize=100))
t_mp = time.time() - t0

assert seq == th == mp, "wszystkie warianty musza dac identyczny wynik"
print(f"Sekwencyjnie:              {t_seq:.3f}s")
print(f"ThreadPool (16 watkow):    {t_thread:.3f}s")
print(f"multiprocessing ({os.cpu_count()} proc.):  {t_mp:.3f}s")

plt.figure(figsize=(7, 4))
plt.bar(["sekwencyjnie", "ThreadPool", "multiprocessing"],
        [t_seq, t_thread, t_mp], color=["#999", "#4caf88", "#3366cc"])
plt.ylabel("czas [s]")
plt.title(f"sentiment_score na {len(texts)} recenzjach")
for i, v in enumerate([t_seq, t_thread, t_mp]):
    plt.text(i, v, f"{v:.2f}s", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(WORKDIR / "concurrency_bench.png", dpi=110)
plt.show()

### Insight 2.1

Kolejność czasów to zwykle **multiprocessing < sekwencyjnie < ThreadPool**. `sentiment_score` jest CPU-bound, więc wątki nie pomagają — GIL pozwala wykonywać bajtkod tylko jednemu wątkowi naraz, a do tego dochodzi narzut na zarządzanie pulą, przez co ThreadPool bywa *wolniejszy* niż wariant sekwencyjny. `multiprocessing.Pool` uruchamia osobne procesy z własnym interpreterem, omija GIL i realnie skaluje się z liczbą rdzeni (`chunksize=100` ogranicza narzut serializacji).

Druga, mniej oczywista lekcja: funkcję `sentiment_score` zapisujemy do osobnego modułu `sentiment.py` i importujemy. Gdyby została zdefiniowana w komórce notatnika, na macOS/Windows (start method `spawn`) `Pool` nie potrafiłby jej zserializować i zadanie by się wywaliło.

---

# Lab 3 — Testowanie

## Teoria w trzech zdaniach

**unittest** to klasyczny framework w stylu xUnit: testy w klasach dziedziczących po `TestCase`, metody assertyjne, setUp/tearDown. **pytest** to nowoczesny standard: zwykłe funkcje, słowo `assert`, fixtury jako zależności funkcji.

Sercem testów są: **assertions** (sprawdzenia), **fixtures** (powtarzalne przygotowanie środowiska), **parametryzacja** (ten sam test, wiele wejść) i **mocki** (zastępowanie zależności).

**Reguła:** test bez asercji to nie test. Test który zależy od kolejności uruchamiania to nie test.

## Przykład rozwiązany: unittest dla `TextStats`

In [ ]:
import unittest
from io import StringIO

class TextStats:
    """Liczy proste statystyki tekstu."""
    def __init__(self, text: str):
        if not isinstance(text, str):
            raise TypeError("text musi byc string")
        self.text = text
    
    def word_count(self) -> int:
        return len(self.text.split())
    
    def char_count(self, with_spaces: bool = True) -> int:
        return len(self.text) if with_spaces else len(self.text.replace(" ", ""))
    
    def avg_word_length(self) -> float:
        words = self.text.split()
        if not words:
            return 0.0
        return sum(len(w) for w in words) / len(words)

class TestTextStats(unittest.TestCase):
    def setUp(self):
        self.empty = TextStats("")
        self.short = TextStats("Pies kot")
        self.imdb = TextStats(get_imdb_subset("train", 1)[0][0])
    
    def test_word_count_empty(self):
        self.assertEqual(self.empty.word_count(), 0)
    
    def test_word_count_short(self):
        self.assertEqual(self.short.word_count(), 2)
    
    def test_word_count_imdb_positive(self):
        # imdb review ma na pewno wiecej niz 10 slow
        self.assertGreater(self.imdb.word_count(), 10)
    
    def test_char_count_with_without_spaces(self):
        self.assertEqual(self.short.char_count(with_spaces=True), 8)
        self.assertEqual(self.short.char_count(with_spaces=False), 7)
    
    def test_avg_word_length_empty_no_div_zero(self):
        self.assertEqual(self.empty.avg_word_length(), 0.0)
    
    def test_type_check(self):
        with self.assertRaises(TypeError):
            TextStats(12345)

# Uruchom w notebooku
runner = unittest.TextTestRunner(stream=StringIO(), verbosity=2)
result = runner.run(unittest.TestLoader().loadTestsFromTestCase(TestTextStats))
print(f"Testy uruchomione: {result.testsRun}")
print(f"Sukces: {result.wasSuccessful()}")
print(f"Bledy: {len(result.errors)}, niepowodzenia: {len(result.failures)}")

## Zadanie 3.1 — pytest dla `Tokenizer` z fixtures + parametrize

**Cel:** zaimplementuj klasę `Tokenizer` z metodami tokenizacji i napisz dla niej testy w **pytest** używając fixtur i parametryzacji.

**Specyfikacja `Tokenizer`:**

```python
class Tokenizer:
    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        ...
    
    def tokenize(self, text: str) -> list[str]:
        # 1. usun tagi HTML jesli strip_html
        # 2. lowercase jesli lower
        # 3. tokeny = regex \w+ 
        # 4. odfiltruj tokeny krotsze niz min_length
        ...
    
    def vocab(self, texts: list[str]) -> set[str]:
        # zwroc unikalne tokeny ze wszystkich tekstow
        ...
```

**Wymagania testowe:**

1. **Fixture** `@pytest.fixture` o nazwie `tokenizer` zwracający `Tokenizer()` z defaultami.
2. **Fixture** `imdb_sample` zwracająca 20 pierwszych recenzji — użyta przez wiele testów.
3. **Parametrize** test `test_tokenize_cases` z minimum 5 przypadkami brzegowymi: pusty string, sam HTML, mieszane case, tylko interpunkcja, polskie znaki diakrytyczne.
4. Test który **musi zawieść** (przykładowo zły flag): oznacz `@pytest.mark.xfail`.
5. Wszystkie testy zapisz w pliku `test_tokenizer.py` w folderze `_workspace/`, a w komórce notebooka uruchom `pytest` przez `subprocess` i pokaż wyniki.

**Insight:** ile średnio unikalnych tokenów jest na 100 recenzji imdb? (heurystyka rozmiaru słownika).

In [ ]:
# Zadanie 3.1 -- Tokenizer + pytest (fixtures, parametrize, xfail)
import subprocess

tokenizer_code = '''
import re

class Tokenizer:
    """Konfigurowalny tokenizator: strip HTML, case folding, filtr dlugosci tokenu."""
    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        self.lower = lower
        self.strip_html = strip_html
        self.min_length = min_length

    def tokenize(self, text: str) -> list[str]:
        """Zamienia tekst na liste tokenow zgodnie z konfiguracja."""
        if self.strip_html:
            text = re.sub(r"<[^>]+>", " ", text)
        if self.lower:
            text = text.lower()
        tokens = re.findall(r"\\w+", text, flags=re.UNICODE)
        return [t for t in tokens if len(t) >= self.min_length]

    def vocab(self, texts: list[str]) -> set[str]:
        """Zwraca zbior unikalnych tokenow ze wszystkich tekstow."""
        out: set[str] = set()
        for t in texts:
            out.update(self.tokenize(t))
        return out
'''

tests_code = '''
import pytest
from tokenizer import Tokenizer

@pytest.fixture
def tokenizer():
    """Default Tokenizer dla wiekszosci testow."""
    return Tokenizer()

@pytest.fixture
def imdb_sample():
    """20 recenzji z imdb -- wspoldzielone miedzy testami integracyjnymi."""
    from datasets import load_dataset
    ds = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(20))
    return [r["text"] for r in ds]

@pytest.mark.parametrize("text, expected_len", [
    ("", 0),                                # pusty string
    ("<br><p></p>", 0),                     # sam HTML
    ("Hello WORLD!", 2),                    # mieszany case
    ("...!?!?!?", 0),                       # tylko interpunkcja
    ("zażółć gęślą jaźń", 3),               # polskie znaki diakrytyczne
    ("the cat sat on the mat", 6),          # zwykle zdanie
])
def test_tokenize_cases(tokenizer, text, expected_len):
    assert len(tokenizer.tokenize(text)) == expected_len

def test_acceptance(tokenizer):
    assert tokenizer.tokenize("<br>Hello WORLD!") == ["hello", "world"]
    assert Tokenizer(lower=False).tokenize("Hello") == ["Hello"]
    assert Tokenizer(strip_html=False).tokenize("<br>hello") == ["br", "hello"]
    assert Tokenizer(min_length=4).tokenize("a bb ccc dddd eeeee") == ["dddd", "eeeee"]

def test_vocab_dedup(tokenizer):
    assert tokenizer.vocab(["aa bb", "bb cc"]) == {"aa", "bb", "cc"}

def test_imdb_integration(tokenizer, imdb_sample):
    """Insight test: 20 recenzji powinno dac sporo unikalnych tokenow."""
    vocab = tokenizer.vocab(imdb_sample)
    assert len(vocab) > 500, f"za malo unikalnych tokenow: {len(vocab)}"

@pytest.mark.xfail(reason="Tokenizer nie wspiera regex z grupowaniem")
def test_advanced_regex_unsupported():
    """Demonstracja xfail -- ten test ma prawo nie zadzialac."""
    tok = Tokenizer()
    assert tok.tokenize("user@domain.com")[0] == "user@domain.com"
'''

(WORKDIR / "tokenizer.py").write_text(tokenizer_code)
(WORKDIR / "test_tokenizer.py").write_text(tests_code)

result = subprocess.run(
    [sys.executable, "-m", "pytest", str(WORKDIR / "test_tokenizer.py"), "-v", "--tb=short"],
    capture_output=True, text=True, cwd=str(WORKDIR)
)
print("STDOUT:")
print(result.stdout[-1500:])
if result.returncode != 0:
    print("\nSTDERR:")
    print(result.stderr[-500:])

# Insight: srednia liczba unikalnych tokenow na 100 recenzji
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
from tokenizer import Tokenizer
vocab_100 = Tokenizer().vocab([t for t, _ in get_imdb_subset("train", 100)])
print(f"\nUnikalnych tokenow na 100 recenzji: {len(vocab_100)}")

### Insight 3.1

`Tokenizer` używa `re.findall(r"\w+", ..., flags=re.UNICODE)`, dzięki czemu poprawnie łapie polskie znaki diakrytyczne (`zażółć gęślą jaźń` → 3 tokeny). Test `test_advanced_regex_unsupported` jest oznaczony `@pytest.mark.xfail` — świadomie dokumentujemy, że tokenizer nie zachowuje adresów e-mail w całości; to nie błąd, lecz zaakceptowane ograniczenie kontraktu.

Liczba unikalnych tokenów na 100 recenzji (rząd kilku tysięcy) rośnie wolniej niż liniowo wraz z liczbą recenzji — to praktyczna ilustracja **prawa Heapsa**: słownik nasyca się, bo nowe recenzje coraz częściej używają słów już widzianych.

---

# Lab 4 — Bazy danych

## Teoria w trzech zdaniach

**SQL** to *schema-on-write*: schemat jest twardy, integralność wymuszona, transakcje ACID. **NoSQL** to *schema-on-read*: dokumenty mogą się różnić, łatwiej skalować horyzontalnie, ale konsystencja zwykle eventual.

**Złota zasada:** wybierasz bazę pod **wzorzec zapytań**, nie pod "jakie mam dane". Jeśli czytasz/piszesz całe dokumenty — NoSQL. Jeśli robisz joiny i agregacje na wymiarach — SQL.

**W SQLite od Pythona 3.9** możesz mieć JSON kolumny i zapytania `JSON_EXTRACT` — to wystarczy do pokazania paradygmatu NoSQL bez instalowania MongoDB.

## Przykład rozwiązany: imdb → SQLite + analityka

In [ ]:
import sqlite3

DB_PATH = WORKDIR / "imdb.db"
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# Schemat -- klasyczna relacja
cur.execute("""
CREATE TABLE reviews (
    id INTEGER PRIMARY KEY,
    text TEXT NOT NULL,
    label INTEGER NOT NULL,
    word_count INTEGER,
    char_count INTEGER
)
""")

# Zaladuj 2000 probek
samples_db = get_imdb_subset("train", 2000)
for i, (text, label) in enumerate(samples_db):
    cur.execute(
        "INSERT INTO reviews (id, text, label, word_count, char_count) VALUES (?, ?, ?, ?, ?)",
        (i, text, label, len(text.split()), len(text))
    )
conn.commit()

# Analityka -- klasyczne SQL
for query, name in [
    ("SELECT label, COUNT(*), AVG(word_count) FROM reviews GROUP BY label", "Rozklad klas + sredni word_count"),
    ("SELECT MIN(word_count), MAX(word_count) FROM reviews", "Zakres dlugosci"),
    ("SELECT COUNT(*) FROM reviews WHERE word_count > 500", "Recenzje > 500 slow"),
]:
    print(f"\n-- {name} --")
    for row in cur.execute(query):
        print(f"  {row}")
conn.close()

## Zadanie 4.1 — NoSQL-style w SQLite (JSON column)

**Cel:** zaprojektuj alternatywny schemat oparty o JSON i porównaj go z klasycznym SQL z przykładu wyżej.

**Wymagania:**

1. Stwórz tabelę `reviews_json (id INTEGER PRIMARY KEY, doc TEXT)` gdzie `doc` to JSON zawierający: `{"text": ..., "label": ..., "stats": {"word_count": ..., "sentiment_hint": "pos"|"neg"}, "tags": [...]}`.
2. Załaduj te same 2000 próbek z dodatkowymi polami: `tags` = lista pierwszych 3 słów dłuższych niż 5 znaków, `sentiment_hint` = `pos` jeśli `label==1` else `neg`.
3. Napisz 4 zapytania w stylu NoSQL używając `json_extract(doc, '$.path')`:
   - Rozkład klas (count per `sentiment_hint`).
   - Średni `word_count` dla każdej klasy.
   - Recenzje gdzie `tags` zawiera słowo "movie" (`LIKE '%movie%'` na JSON).
   - Top 5 najdłuższych recenzji w klasie pozytywnej.
4. **Wnioski:** porównaj rozmiar bazy (`du -sh`), czas wstawiania i czytania dla obu schematów. Który schemat jest lepszy dla *tego* problemu i dlaczego?

In [ ]:
# Zadanie 4.1 -- NoSQL style w SQLite (kolumna JSON)

DB_JSON = WORKDIR / "imdb_json.db"
if DB_JSON.exists():
    DB_JSON.unlink()

conn2 = sqlite3.connect(str(DB_JSON))
cur2 = conn2.cursor()

cur2.execute("""
CREATE TABLE reviews_json (
    id INTEGER PRIMARY KEY,
    doc TEXT NOT NULL
)
""")

samples_nosql = get_imdb_subset("train", 2000)

t0 = time.time()
for i, (text, label) in enumerate(samples_nosql):
    words = text.split()
    doc = {
        "text": text,
        "label": label,
        "stats": {"word_count": len(words), "sentiment_hint": "pos" if label == 1 else "neg"},
        "tags": [w for w in words if len(w) > 5][:3],
    }
    cur2.execute("INSERT INTO reviews_json (id, doc) VALUES (?, ?)", (i, json.dumps(doc)))
conn2.commit()
insert_time_json = time.time() - t0

queries = {
    "rozklad_klas": """
        SELECT json_extract(doc, '$.stats.sentiment_hint') AS hint, COUNT(*) AS n
        FROM reviews_json
        GROUP BY hint
    """,
    "avg_word_count_per_class": """
        SELECT json_extract(doc, '$.stats.sentiment_hint') AS hint,
               ROUND(AVG(json_extract(doc, '$.stats.word_count')), 1) AS avg_wc
        FROM reviews_json
        GROUP BY hint
    """,
    "tags_zawiera_movie": """
        SELECT COUNT(*) AS n
        FROM reviews_json
        WHERE lower(json_extract(doc, '$.tags')) LIKE '%movie%'
    """,
    "top5_najdluzsze_pozytywne": """
        SELECT id, json_extract(doc, '$.stats.word_count') AS wc
        FROM reviews_json
        WHERE json_extract(doc, '$.label') = 1
        ORDER BY wc DESC
        LIMIT 5
    """,
}

t0 = time.time()
for name, sql in queries.items():
    print(f"\n-- {name} --")
    for row in cur2.execute(sql):
        print(f"  {row}")
read_time_json = time.time() - t0

size_sql = os.path.getsize(DB_PATH) if DB_PATH.exists() else 0
size_json = os.path.getsize(DB_JSON)
print(f"\n=== Porownanie schematow ===")
print(f"SQL  (reviews):       {size_sql:>10,} B")
print(f"JSON (reviews_json):  {size_json:>10,} B  ({size_json / max(size_sql, 1):.2f}x SQL)")
print(f"JSON insert 2000:     {insert_time_json:.3f}s")
print(f"JSON read (4 zapytania): {read_time_json:.4f}s")

conn2.close()

### Insight 4.1

Schemat JSON (`reviews_json`) jest **nieco większy** od klasycznego SQL, bo w każdym wierszu powtarza nazwy kluczy (`text`, `label`, `stats`, ...) — to cena za elastyczność *schema-on-read*. Zapytania przez `json_extract` działają, ale bez indeksów na ścieżkach JSON są wolniejsze niż na typowanych kolumnach SQL.

Subtelność w zapytaniu `tags_zawiera_movie`: `tags` to lista słów **dłuższych niż 5 znaków**, więc dokładne słowo `"movie"` (5 znaków) nigdy tam nie trafi — ale `LIKE '%movie%'` łapie warianty typu `movies`, `moviegoer`, które już mają >5 znaków. Dla *tego* problemu (stały, znany schemat rekordu, dużo agregacji po wymiarach) **klasyczny SQL jest lepszy**; JSON column ma sens, gdy struktura rekordu jest zmienna lub rzadka.

---

# Lab 5 — PySpark

## Teoria w trzech zdaniach

**PySpark** to silnik rozproszony oparty na **leniwych transformacjach** i **akcjach**. Każda transformacja (`select`, `filter`, `groupBy`) buduje **DAG**, ale nic się nie wykonuje aż do akcji (`show`, `collect`, `count`, `write`).

**Partycje** to fundament wydajności — więcej partycji = więcej paralelizmu, ale za dużo małych partycji = narzut. Reguła kciuka: 2-4 partycje na rdzeń CPU.

**Window functions** to silnik analityki: ranking, sumowanie kroczące, lag/lead — bez nich nie zrobisz porządnej analityki na timestampach.

## Przykład rozwiązany: imdb → Spark + count słów per klasa

In [ ]:
from pyspark.sql import SparkSession, functions as F

# Setup Sparka -- robust
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
for candidate in [
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home",
    "/usr/lib/jvm/java-17-openjdk-amd64",
]:
    if os.path.exists(candidate):
        os.environ.setdefault("JAVA_HOME", candidate)
        break

spark = (SparkSession.builder
    .appName("AAP zaliczenie")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready")

# Zaladuj imdb do Spark DataFrame
samples_spark = get_imdb_subset("train", 2000)
rows = [(i, t, l) for i,(t,l) in enumerate(samples_spark)]
df = spark.createDataFrame(rows, ["id", "text", "label"])

# Liczba slow per klasa
df_words = (df
    .withColumn("words", F.split(F.lower(F.regexp_replace("text", r"<[^>]+>", " ")), r"\W+"))
    .withColumn("word_count", F.size("words")))

print("\n-- Statystyki per klasa --")
df_words.groupBy("label").agg(
    F.count("*").alias("n"),
    F.round(F.avg("word_count"), 1).alias("avg_words"),
    F.expr("percentile_approx(word_count, 0.5)").alias("median_words")
).show()

# Najczestsze slowa per klasa (top 10 pozytywne)
df_exploded = df_words.select("label", F.explode("words").alias("word"))
df_exploded = df_exploded.filter((F.length("word") > 3) & (F.col("label") == 1))
print("\n-- Top 10 slow w pozytywnych recenzjach --")
df_exploded.groupBy("word").count().orderBy(F.col("count").desc()).limit(10).show()

## Zadanie 5.1 — Window functions: ranking recenzji

**Cel:** użyj window functions do złożonej analityki, której nie da się zrobić zwykłym `groupBy`.

**Wymagania:**

1. Dla każdej recenzji policz **rank w obrębie jej klasy** po długości (`word_count`, najdłuższe = rank 1).
2. Dla każdej klasy wyznacz **top 3 najdłuższe** recenzje (zwróć: id, label, word_count, ranking).
3. Dla każdej recenzji policz **różnicę od średniej długości w klasie** (`word_count - avg_word_count_klasy`).
4. **Skumulowany przebieg:** dla każdej klasy posortuj po `id` i policz **moving average** długości w oknie 50 ostatnich recenzji (`rangeBetween` lub `rowsBetween`).
5. Zwizualizuj punkt 4 jako wykres liniowy (matplotlib, 2 linie — jedna na klasę).

**Wskazówka:** użyj `pyspark.sql.Window`:

```python
from pyspark.sql.window import Window
w = Window.partitionBy("label").orderBy(F.col("word_count").desc())
df.withColumn("rank", F.row_number().over(w))
```

In [ ]:
# Zadanie 5.1 -- window functions: ranking i analityka recenzji
from pyspark.sql.window import Window

# 1. Ranking dlugosci w obrebie klasy (najdluzsze = rank 1)
w_rank = Window.partitionBy("label").orderBy(F.col("word_count").desc())
ranked = df_words.withColumn("rank", F.row_number().over(w_rank))

# 2. Top 3 najdluzsze recenzje per klasa
print("-- Top 3 najdluzsze recenzje w kazdej klasie --")
ranked.filter(F.col("rank") <= 3) \
      .select("id", "label", "word_count", "rank") \
      .orderBy("label", "rank") \
      .show()

# 3. Roznica dlugosci od sredniej w klasie
w_class = Window.partitionBy("label")
diff = (df_words
        .withColumn("avg_wc_class", F.avg("word_count").over(w_class))
        .withColumn("diff_from_avg", F.round(F.col("word_count") - F.col("avg_wc_class"), 1)))
print("-- Roznica od sredniej klasowej (10 przykladow) --")
diff.select("id", "label", "word_count", "diff_from_avg").show(10)

# 4. Moving average dlugosci w oknie 50 ostatnich recenzji (po id, w klasie)
w_move = Window.partitionBy("label").orderBy("id").rowsBetween(-49, 0)
moving = df_words.withColumn("moving_avg_wc", F.round(F.avg("word_count").over(w_move), 1))

# 5. Wykres liniowy moving average -- jedna linia na klase
pdf_move = moving.select("id", "label", "moving_avg_wc").orderBy("id").toPandas()

plt.figure(figsize=(10, 5))
for lab, grp in pdf_move.groupby("label"):
    plt.plot(grp["id"], grp["moving_avg_wc"], label=f"label={lab}", linewidth=1)
plt.xlabel("id recenzji")
plt.ylabel("srednia krocząca dlugosci (okno 50)")
plt.title("Moving average dlugosci recenzji wg klasy")
plt.legend()
plt.tight_layout()
plt.savefig(WORKDIR / "moving_avg_wc.png", dpi=110)
plt.show()

### Insight 5.1

Window functions robią to, czego `groupBy` nie potrafi: liczą wartość *per wiersz* z zachowaniem kontekstu grupy. `row_number()` daje ranking bez kolapsowania wierszy, a `rowsBetween(-49, 0)` liczy średnią kroczącą z 50 ostatnich rekordów w obrębie klasy. Kluczowa pułapka wydajnościowa: każde okno z `orderBy` wymusza **shuffle + sort** po kolumnie partycjonującej, dlatego ustawiliśmy `spark.sql.shuffle.partitions=4` (mały zbiór — domyślne 200 partycji to czysty narzut). Wykres pokazuje, że średnia długość recenzji bywa zauważalnie inna między klasami pos/neg.

---

# Lab 6 — Data Quality (jakość danych)

## Teoria w trzech zdaniach

**Data Quality** to nie audyt po wszystkim — to **kontrakt** który dane muszą spełnić zanim wejdą do pipeline'u. Sześć wymiarów: kompletność, unikalność, poprawność, zgodność, świeżość, integralność.

Współczesny stack: `pandera`/`great_expectations` dla **deklaratywnych testów**, `pandas-profiling` (teraz `ydata-profiling`) dla **raportów eksploracyjnych**, własne **walidatory** dla reguł biznesowych.

**Reguła:** jeśli nie potrafisz w jednym zdaniu opisać co znaczy "dobre dane" dla Twojego problemu, nie powinieneś jeszcze trenować modelu.

## Przykład rozwiązany: profilowanie imdb — wykrywanie anomalii

In [ ]:
import pandas as pd

# Wczytaj wieksza probke
samples_dq = get_imdb_subset("train", 2000)
df_pd = pd.DataFrame(samples_dq, columns=["text", "label"])
df_pd["word_count"] = df_pd["text"].str.split().str.len()
df_pd["char_count"] = df_pd["text"].str.len()

# Profil podstawowy
print("=== KOMPLETNOSC ===")
nulls = df_pd.isnull().sum()
print(f"Nulle: {dict(nulls)}")

print("\n=== UNIKALNOSC ===")
dup_count = df_pd["text"].duplicated().sum()
print(f"Duplikaty tekstu: {dup_count}")

print("\n=== ROZKLAD LABELI ===")
print(df_pd["label"].value_counts(normalize=True).rename("frac"))
balance_ratio = df_pd["label"].value_counts().min() / df_pd["label"].value_counts().max()
print(f"Stosunek mniejszosci do wiekszosci: {balance_ratio:.3f} (1.0 = idealnie zbalansowane)")

print("\n=== ANOMALIE DLUGOSCI ===")
p99 = df_pd["word_count"].quantile(0.99)
p01 = df_pd["word_count"].quantile(0.01)
outliers = df_pd[(df_pd["word_count"] > p99) | (df_pd["word_count"] < p01)]
print(f"P1: {p01:.0f}, P99: {p99:.0f}, outlierow (poza P1-P99): {len(outliers)}")

print("\n=== ANOMALIE TRESCI ===")
has_html = df_pd["text"].str.contains(r"<[^>]+>", regex=True).sum()
very_short = (df_pd["word_count"] < 5).sum()
print(f"Tekst zawiera HTML tagi: {has_html} ({has_html/len(df_pd)*100:.1f}%)")
print(f"Bardzo krotkie recenzje (<5 slow): {very_short}")

print("\nINSIGHT: imdb ma duzo HTML pozostalosci (<br />). Trzeba je czyscic przed treningiem!")

## Zadanie 6.1 — Kontrakt danych + raport JSON

**Cel:** zaimplementuj prosty *Data Quality Framework* w czystym Pythonie i wygeneruj raport o jakości datasetu.

**Wymagania:**

1. Klasa `DataContract` z metodą `add_rule(name, callable, severity)` (severity ∈ {`info`, `warning`, `error`}).
2. Klasa `DataValidator` która iteruje po regułach kontraktu i zwraca raport: `{rule_name: {passed: bool, severity, details}}`.
3. Zdefiniuj kontrakt dla imdb z **minimum 6 regułami**:
   - `no_nulls` — brak NULL w `text` i `label`
   - `labels_in_set` — wszystkie labele są w {0, 1}
   - `min_word_count` — każda recenzja ma min. 5 słów
   - `max_word_count` — żadna recenzja > 2000 słów (sanity)
   - `no_duplicates` — brak duplikatów `text`
   - `class_balance` — stosunek klas między 0.5 a 1.5
4. Reguły o severity `error` które zawiodły powinny rzucić wyjątek (fail fast). Reszta jest tylko ostrzeżeniem.
5. Wygeneruj raport w pliku `_workspace/data_quality_report.json` z timestampem.

**Bonus:** zaimplementuj `severity="warning"` regułę `no_html_tags` i pokaż że *raport* o niej mówi, ale walidacja nie zawodzi.

In [ ]:
# Zadanie 6.1 -- DataContract + DataValidator + raport JSON
from datetime import datetime
from dataclasses import dataclass
from typing import Callable

@dataclass
class Rule:
    name: str
    check: Callable
    severity: str = "warning"  # info | warning | error

class DataContract:
    """Zbior nazwanych regul jakosci danych z przypisanym poziomem severity."""
    def __init__(self, name: str):
        self.name = name
        self.rules: list[Rule] = []

    def add_rule(self, name: str, check: Callable, severity: str = "warning") -> None:
        """Dodaje regule do kontraktu."""
        self.rules.append(Rule(name, check, severity))

class DataValidator:
    """Uruchamia reguly kontraktu; reguly 'error' przerywaja walidacje (fail fast)."""
    def __init__(self, contract: DataContract):
        self.contract = contract

    def validate(self, df) -> dict:
        """Zwraca raport {nazwa: {passed, severity, details}}; rzuca przy bledzie severity=error."""
        report = {}
        for rule in self.contract.rules:
            try:
                passed = bool(rule.check(df))
                details = "OK" if passed else "regula niespelniona"
            except Exception as e:
                passed, details = False, f"blad wykonania: {e}"
            report[rule.name] = {"passed": passed, "severity": rule.severity, "details": details}
            if rule.severity == "error" and not passed:
                raise ValueError(f"Reguła '{rule.name}' (error) niespelniona: {details}")
        return report

def _balance(d) -> float:
    counts = d["label"].value_counts()
    return counts.min() / counts.max()

contract = DataContract("imdb_quality")
contract.add_rule("no_nulls",      lambda d: d[["text", "label"]].notnull().all().all(), "error")
contract.add_rule("labels_in_set", lambda d: set(d["label"].unique()) <= {0, 1},          "error")
contract.add_rule("min_word_count", lambda d: (d["word_count"] >= 5).all(),                "warning")
contract.add_rule("max_word_count", lambda d: (d["word_count"] <= 2000).all(),             "warning")
contract.add_rule("no_duplicates",  lambda d: not d["text"].duplicated().any(),            "warning")
contract.add_rule("class_balance",  lambda d: 0.5 <= _balance(d) <= 1.5,                    "warning")
contract.add_rule("no_html_tags",   lambda d: not d["text"].str.contains(r"<[^>]+>", regex=True).any(), "warning")

report = DataValidator(contract).validate(df_pd)
print("=== Raport jakosci danych ===")
for name, res in report.items():
    flaga = "OK " if res["passed"] else "!! "
    print(f"  [{flaga}] {name:<16} ({res['severity']}) -> {res['details']}")

raport_full = {
    "contract": contract.name,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "n_rows": int(len(df_pd)),
    "results": report,
}
report_path = WORKDIR / "data_quality_report.json"
report_path.write_text(json.dumps(raport_full, indent=2, ensure_ascii=False))
print(f"\nRaport zapisany: {report_path}")

### Insight 6.1

Rozdzielenie `DataContract` (deklaracja reguł) od `DataValidator` (egzekucja) to wprost zasada *open/closed*: nowe reguły dodajemy bez dotykania silnika walidacji. Mechanizm severity oddaje realny pipeline produkcyjny — reguły `error` (np. `no_nulls`, `labels_in_set`) zatrzymują przetwarzanie natychmiast (*fail fast*), a `warning` (np. `no_html_tags`) tylko trafiają do raportu i nie blokują. To różnica między **audytem** (raport po fakcie) a **kontraktem** (bramka, której dane muszą dotrzymać, zanim wejdą do modelu). Raport zapisujemy do JSON z timestampem, żeby dało się go wersjonować i porównywać między uruchomieniami.

---

# Sekcja kontrolna — co umiesz po tym zestawie

Po ukończeniu wszystkich 6 zadań powinieneś **bez przygotowania** odpowiedzieć na:

1. **Dekorator:** kiedy `functools.wraps` jest konieczny, a kiedy można sobie odpuścić?
2. **Concurrency:** dlaczego threading nie przyspieszy obliczeń, a multiprocessing przyspieszy?
3. **Testowanie:** kiedy lepiej parametrize, a kiedy osobne testy?
4. **Bazy:** co znaczy *schema-on-read*? Daj praktyczny przykład gdy to plus, a kiedy minus.
5. **Spark:** co to znaczy że transformacja jest "lazy"? Daj przykład **kiedy to boli** w debugowaniu.
6. **Data Quality:** różnica między *audytem* a *kontraktem* danych. W produkcji potrzebujesz obu — dlaczego?

## Co dalej?

- **Pakowanie:** `pyproject.toml`, `poetry`, dystrybucja przez `pip`
- **CI/CD:** GitHub Actions, pre-commit hooks, automated testing
- **Observability:** logging structured (`structlog`), metryki (`prometheus_client`), tracing (`opentelemetry`)
- **Orchestration:** Apache Airflow / Prefect / Dagster do *prawdziwych* pipeline'ów
- **Workshops:** [Real Python](https://realpython.com), [Talk Python To Me](https://talkpython.fm) podcast

In [ ]:
# Sprzatanie
try:
    spark.stop()
    print("Spark zatrzymany.")
except NameError:
    pass
print(f"Workspace: {WORKDIR.resolve()}")
print("Wszystkie cache i artefakty zostaja -- usun recznie jesli potrzeba.")